In [1]:
# ORB = > Oriented FAST and Rotated BRIEF => feature detector + descriptor
import cv2
import numpy as np

In [5]:
cap = cv2.VideoCapture(0)
ret, prev = cap.read()

orb = cv2.ORB_create(300)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck = True)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    kp1, des1 = orb.detectAndCompute(prev, None)
    kp2, des2 = orb.detectAndCompute(frame, None)
    out = frame.copy()
    if  des1 is not None and des2 is not None:
        matches = sorted(bf.match(des1,des2), key = lambda m:m.distance)[:50]
        vis = cv2.drawMatches(prev, kp1, frame, kp2, matches, None, flags = 2)
        cv2.imshow("feature matches", vis) 
       
    cv2.imshow("current", out)
    if cv2.waitKey(1)&0xFF ==27:
        break
    
cap.release()
cv2.destroyAllWindows()

In [ ]:
# # smoothening

# 1.start webcam
# 2. cap first frame
# 3. cov to grayscale
# 4. detect the feature point
# 5. cap next frame
# 6. optical flow tack the old
# 7. remove -> track
# 8. are u having point => if not use the original
#     estimate affine tran
#     extract dx, dy and rotation 
#     update the camera trajectory
#     cal motion correction
#     warp current frame
#     display the stabilized frame
#     detect new feature point
#     repeat


# => video stabilization
#     horiz shaking
#     verti shaking
#     minor move caused by walking

In [20]:
import cv2
import numpy as np


# ==================================================
# SETTINGS
# ==================================================

input_video = "stabilize.mp4"
output_video = "stabilized_output.mp4"

SMOOTHING_RADIUS = 30


# ==================================================
# 1. OPEN VIDEO
# ==================================================

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise RuntimeError("Cannot open video.")


n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

print("Frames:", n_frames)
print("Resolution:", w, "x", h)
print("FPS:", fps)


# ==================================================
# 2. READ FIRST FRAME
# ==================================================

ret, prev = cap.read()

if not ret:
    raise RuntimeError("Cannot read first frame.")

prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)


# Store frame-to-frame transformations
transforms = np.zeros((n_frames - 1, 3), np.float32)


# ==================================================
# 3. FIRST PASS
#    CALCULATE CAMERA MOTION
# ==================================================

for i in range(n_frames - 1):

    ret, curr = cap.read()

    if not ret:
        break

    curr_gray = cv2.cvtColor(
        curr,
        cv2.COLOR_BGR2GRAY
    )


    # Detect feature points
    prev_pts = cv2.goodFeaturesToTrack(
        prev_gray,
        maxCorners=300,
        qualityLevel=0.01,
        minDistance=20,
        blockSize=3
    )


    if prev_pts is None:

        transforms[i] = [0, 0, 0]
        prev_gray = curr_gray
        continue


    # Track feature points
    curr_pts, status, err = cv2.calcOpticalFlowPyrLK(
        prev_gray,
        curr_gray,
        prev_pts,
        None
    )


    if curr_pts is None or status is None:

        transforms[i] = [0, 0, 0]
        prev_gray = curr_gray
        continue


    # Keep successfully tracked points
    good_old = prev_pts[
        status.flatten() == 1
    ]

    good_new = curr_pts[
        status.flatten() == 1
    ]


    if len(good_old) < 8:

        transforms[i] = [0, 0, 0]
        prev_gray = curr_gray
        continue


    # Estimate transformation
    M, inliers = cv2.estimateAffinePartial2D(
        good_old,
        good_new
    )


    if M is None:

        transforms[i] = [0, 0, 0]

    else:

        dx = M[0, 2]
        dy = M[1, 2]

        da = np.arctan2(
            M[1, 0],
            M[0, 0]
        )

        transforms[i] = [
            dx,
            dy,
            da
        ]


    print(
        f"Processing frame {i + 1}/{n_frames - 1}",
        end="\r"
    )

    prev_gray = curr_gray


cap.release()


# ==================================================
# 4. CALCULATE CAMERA TRAJECTORY
# ==================================================

trajectory = np.cumsum(
    transforms,
    axis=0
)


# ==================================================
# 5. MOVING AVERAGE SMOOTHING
# ==================================================

def moving_average(curve, radius):

    window_size = 2 * radius + 1

    # Pad the beginning and end
    curve_pad = np.pad(
        curve,
        (radius, radius),
        mode="edge"
    )

    # Moving average
    kernel = np.ones(
        window_size
    ) / window_size

    curve_smoothed = np.convolve(
        curve_pad,
        kernel,
        mode="same"
    )

    # Remove padding
    curve_smoothed = curve_smoothed[
        radius:-radius
    ]

    return curve_smoothed


def smooth_trajectory(trajectory):

    smoothed = np.copy(
        trajectory
    )

    for i in range(3):

        smoothed[:, i] = moving_average(
            trajectory[:, i],
            SMOOTHING_RADIUS
        )

    return smoothed


smooth_trajectory_data = smooth_trajectory(
    trajectory
)


# ==================================================
# 6. CALCULATE STABILIZATION CORRECTION
# ==================================================

difference = (
    smooth_trajectory_data
    - trajectory
)

transforms_smooth = (
    transforms
    + difference
)


# ==================================================
# 7. OPEN VIDEO AGAIN
# ==================================================

cap = cv2.VideoCapture(
    input_video
)


# Output writer
fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)

out = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (w, h)
)


# ==================================================
# 8. STABILIZE EACH FRAME
# ==================================================

ret, frame = cap.read()

if ret:

    out.write(frame)


for i in range(n_frames - 1):

    ret, frame = cap.read()

    if not ret:
        break


    # Get corrected transformation
    dx = transforms_smooth[i, 0]

    dy = transforms_smooth[i, 1]

    da = transforms_smooth[i, 2]


    # Create transformation matrix
    cos = np.cos(da)

    sin = np.sin(da)


    M = np.array(
        [
            [cos, -sin, dx],
            [sin,  cos, dy]
        ],
        dtype=np.float32
    )


    # Apply stabilization
    stabilized = cv2.warpAffine(
        frame,
        M,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REPLICATE
    )


    # ----------------------------------------------
    # ZOOM TO REMOVE BORDER MOVEMENT
    # ----------------------------------------------

    zoom = 1.05

    crop_w = int(w / zoom)
    crop_h = int(h / zoom)

    x = (w - crop_w) // 2
    y = (h - crop_h) // 2


    stabilized = stabilized[
        y:y + crop_h,
        x:x + crop_w
    ]


    stabilized = cv2.resize(
        stabilized,
        (w, h)
    )


    # Save stabilized frame
    out.write(
        stabilized
    )


    # ----------------------------------------------
    # DISPLAY COMPARISON
    # ----------------------------------------------

    original_display = frame.copy()

    cv2.putText(
        original_display,
        "Original",
        (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    cv2.putText(
        stabilized,
        "Stabilized",
        (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )


    comparison = np.hstack(
        (
            original_display,
            stabilized
        )
    )


    # Resize display if screen is small
    display = cv2.resize(
        comparison,
        (1280, 720)
    )


    cv2.imshow(
        "Original | Stabilized",
        display
    )


    print(
        f"Stabilizing frame {i + 1}/{n_frames - 1}",
        end="\r"
    )


    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# ==================================================
# 9. RELEASE
# ==================================================

cap.release()

out.release()

cv2.destroyAllWindows()

print("\nDone!")
print("Saved as:", output_video)

Frames: 158
Resolution: 650 x 464
FPS: 25.315779635217776
Stabilizing frame 157/157
Done!
Saved as: stabilized_output.mp4


In [24]:
import cv2
import numpy as np


# ============================================================
# SETTINGS
# ============================================================

INPUT_VIDEO = "stabilize.mp4"

STABILIZED_OUTPUT = "stabilized_output.mp4"
COMPARISON_OUTPUT = "comparison_output.mp4"

SMOOTHING_RADIUS = 30
ZOOM = 1.08

MAX_CORNERS = 300
QUALITY_LEVEL = 0.01
MIN_DISTANCE = 20


# ============================================================
# FUNCTION: SMOOTH CURVE
# ============================================================

def moving_average(curve, radius):

    window_size = 2 * radius + 1

    # Pad the curve at both ends
    curve_pad = np.pad(
        curve,
        (radius, radius),
        mode="edge"
    )

    # Create averaging kernel
    kernel = np.ones(window_size) / window_size

    # Apply moving average
    curve_smoothed = np.convolve(
        curve_pad,
        kernel,
        mode="same"
    )

    # Remove padding
    curve_smoothed = curve_smoothed[
        radius:-radius
    ]

    return curve_smoothed


# ============================================================
# FUNCTION: SMOOTH CAMERA TRAJECTORY
# ============================================================

def smooth_trajectory(trajectory, radius):

    smoothed = np.copy(trajectory)

    # Smooth X, Y and angle separately
    for i in range(3):

        smoothed[:, i] = moving_average(
            trajectory[:, i],
            radius
        )

    return smoothed


# ============================================================
# FUNCTION: FIX BORDER BY ZOOMING
# ============================================================

def fix_border(frame, zoom):

    h, w = frame.shape[:2]

    crop_w = int(w / zoom)
    crop_h = int(h / zoom)

    x = (w - crop_w) // 2
    y = (h - crop_h) // 2

    cropped = frame[
        y:y + crop_h,
        x:x + crop_w
    ]

    return cv2.resize(
        cropped,
        (w, h),
        interpolation=cv2.INTER_LINEAR
    )


# ============================================================
# 1. OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(INPUT_VIDEO)

if not cap.isOpened():
    raise RuntimeError(
        f"Cannot open video: {INPUT_VIDEO}"
    )


# Get video properties
n_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

w = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

h = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

fps = cap.get(
    cv2.CAP_PROP_FPS
)


print("Video information")
print("-----------------")
print("Frames:", n_frames)
print("Width:", w)
print("Height:", h)
print("FPS:", fps)


# ============================================================
# 2. READ FIRST FRAME
# ============================================================

ret, prev = cap.read()

if not ret:

    cap.release()

    raise RuntimeError(
        "Cannot read first frame."
    )


prev_gray = cv2.cvtColor(
    prev,
    cv2.COLOR_BGR2GRAY
)


# ============================================================
# 3. STORE FRAME-TO-FRAME MOTIONS
# ============================================================

transforms = np.zeros(
    (n_frames - 1, 3),
    dtype=np.float32
)


# ============================================================
# 4. FIRST PASS
#    CALCULATE CAMERA MOTION
# ============================================================

print("\nCalculating camera motion...")


for i in range(n_frames - 1):

    ret, curr = cap.read()

    if not ret:
        break


    curr_gray = cv2.cvtColor(
        curr,
        cv2.COLOR_BGR2GRAY
    )


    # --------------------------------------------------------
    # DETECT FEATURE POINTS
    # --------------------------------------------------------

    prev_pts = cv2.goodFeaturesToTrack(
        prev_gray,
        maxCorners=MAX_CORNERS,
        qualityLevel=QUALITY_LEVEL,
        minDistance=MIN_DISTANCE,
        blockSize=3
    )


    # If no features found
    if prev_pts is None:

        transforms[i] = [0, 0, 0]

        prev_gray = curr_gray

        continue


    # --------------------------------------------------------
    # TRACK FEATURE POINTS
    # --------------------------------------------------------

    curr_pts, status, error = cv2.calcOpticalFlowPyrLK(
        prev_gray,
        curr_gray,
        prev_pts,
        None
    )


    # Check optical flow result
    if curr_pts is None or status is None:

        transforms[i] = [0, 0, 0]

        prev_gray = curr_gray

        continue


    # --------------------------------------------------------
    # KEEP SUCCESSFULLY TRACKED POINTS
    # --------------------------------------------------------

    good_old = prev_pts[
        status.flatten() == 1
    ]

    good_new = curr_pts[
        status.flatten() == 1
    ]


    # Need enough points
    if len(good_old) < 8:

        transforms[i] = [0, 0, 0]

        prev_gray = curr_gray

        continue


    # --------------------------------------------------------
    # ESTIMATE CAMERA TRANSFORMATION
    # --------------------------------------------------------

    M, inliers = cv2.estimateAffinePartial2D(
        good_old,
        good_new
    )


    if M is None:

        transforms[i] = [0, 0, 0]


    else:

        # Translation
        dx = M[0, 2]
        dy = M[1, 2]


        # Rotation
        da = np.arctan2(
            M[1, 0],
            M[0, 0]
        )


        transforms[i] = [
            dx,
            dy,
            da
        ]


    # Update previous frame
    prev_gray = curr_gray


    print(
        f"Motion: {i + 1}/{n_frames - 1}",
        end="\r"
    )


cap.release()


print("\nCamera motion calculation completed.")


# ============================================================
# 5. CALCULATE CAMERA TRAJECTORY
# ============================================================

trajectory = np.cumsum(
    transforms,
    axis=0
)


# ============================================================
# 6. SMOOTH CAMERA TRAJECTORY
# ============================================================

smooth_trajectory_data = smooth_trajectory(
    trajectory,
    SMOOTHING_RADIUS
)


# ============================================================
# 7. CALCULATE STABILIZATION CORRECTION
# ============================================================

difference = (
    smooth_trajectory_data
    - trajectory
)


transforms_smooth = (
    transforms
    + difference
)


# ============================================================
# 8. OPEN VIDEO AGAIN
#    SECOND PASS
# ============================================================

cap = cv2.VideoCapture(INPUT_VIDEO)

if not cap.isOpened():

    raise RuntimeError(
        "Cannot reopen video."
    )


# ============================================================
# 9. CREATE OUTPUT VIDEO WRITERS
# ============================================================

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)


# Stabilized video
out_stabilized = cv2.VideoWriter(
    STABILIZED_OUTPUT,
    fourcc,
    fps,
    (w, h)
)


# Side-by-side comparison video
out_comparison = cv2.VideoWriter(
    COMPARISON_OUTPUT,
    fourcc,
    fps,
    (w * 2, h)
)


# ============================================================
# 10. READ AND SAVE FIRST FRAME
# ============================================================

ret, first_frame = cap.read()

if not ret:

    cap.release()

    raise RuntimeError(
        "Cannot read first frame."
    )


# First frame has no stabilization transform
first_stabilized = fix_border(
    first_frame,
    ZOOM
)


# Add labels
first_original_display = first_frame.copy()

cv2.putText(
    first_original_display,
    "Original",
    (30, 50),
    cv2.FONT_HERSHEY_SIMPLEX,
    1,
    (0, 255, 0),
    2
)

cv2.putText(
    first_stabilized,
    "Stabilized",
    (30, 50),
    cv2.FONT_HERSHEY_SIMPLEX,
    1,
    (0, 255, 0),
    2
)


# Side-by-side first frame
first_comparison = np.hstack(
    (
        first_original_display,
        first_stabilized
    )
)


out_stabilized.write(
    first_stabilized
)

out_comparison.write(
    first_comparison
)


# ============================================================
# 11. PROCESS REMAINING FRAMES
# ============================================================

print("\nStabilizing video...")


for i in range(n_frames - 1):

    ret, frame = cap.read()

    if not ret:
        break


    # --------------------------------------------------------
    # GET CORRECTED MOTION
    # --------------------------------------------------------

    dx = transforms_smooth[i, 0]

    dy = transforms_smooth[i, 1]

    da = transforms_smooth[i, 2]


    # --------------------------------------------------------
    # CREATE TRANSFORMATION MATRIX
    # --------------------------------------------------------

    cos = np.cos(da)
    sin = np.sin(da)


    M = np.array(
        [
            [cos, -sin, dx],
            [sin,  cos, dy]
        ],
        dtype=np.float32
    )


    # --------------------------------------------------------
    # APPLY STABILIZATION
    # --------------------------------------------------------

    stabilized = cv2.warpAffine(
        frame,
        M,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REPLICATE
    )


    # --------------------------------------------------------
    # FIX BLACK / MOVING BORDERS
    # --------------------------------------------------------

    stabilized = fix_border(
        stabilized,
        ZOOM
    )


    # --------------------------------------------------------
    # CREATE ORIGINAL DISPLAY
    # --------------------------------------------------------

    original_display = frame.copy()


    # Add labels
    cv2.putText(
        original_display,
        "Original",
        (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )


    cv2.putText(
        stabilized,
        "Stabilized",
        (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )


    # --------------------------------------------------------
    # COMBINE SIDE BY SIDE
    # --------------------------------------------------------

    comparison = np.hstack(
        (
            original_display,
            stabilized
        )
    )


    # --------------------------------------------------------
    # SAVE OUTPUTS
    # --------------------------------------------------------

    out_stabilized.write(
        stabilized
    )

    out_comparison.write(
        comparison
    )


    # --------------------------------------------------------
    # DISPLAY SIDE BY SIDE
    # --------------------------------------------------------

    display_width = 1280

    display_height = int(
        comparison.shape[0]
        * display_width
        / comparison.shape[1]
    )


    display = cv2.resize(
        comparison,
        (display_width, display_height)
    )


    cv2.imshow(
        "Original | Stabilized",
        display
    )


    print(
        f"Stabilizing: {i + 1}/{n_frames - 1}",
        end="\r"
    )


    # Press Q to stop
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# ============================================================
# 12. RELEASE EVERYTHING
# ============================================================

cap.release()

out_stabilized.release()

out_comparison.release()

cv2.destroyAllWindows()


# ============================================================
# DONE
# ============================================================

print("\n\nDone!")

print("Stabilized video saved as:")
print(STABILIZED_OUTPUT)

print("\nSide-by-side comparison saved as:")
print(COMPARISON_OUTPUT)

Video information
-----------------
Frames: 158
Width: 650
Height: 464
FPS: 25.315779635217776

Calculating camera motion...
Motion: 157/157
Camera motion calculation completed.

Stabilizing video...
Stabilizing: 157/157

Done!
Stabilized video saved as:
stabilized_output.mp4

Side-by-side comparison saved as:
comparison_output.mp4


In [27]:
import cv2
import numpy as np

video = "stabilize.mp4"
radius = 30

# ---------------------------------
# SMOOTHING FUNCTION
# ---------------------------------

def smooth(curve, radius):
    curve = np.pad(curve, (radius, radius), mode="edge")
    window = np.ones(2 * radius + 1) / (2 * radius + 1)
    return np.convolve(curve, window, mode="same")[radius:-radius]


# ---------------------------------
# FIRST PASS: FIND CAMERA MOTION
# ---------------------------------

cap = cv2.VideoCapture(video)

n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

ret, prev = cap.read()

if not ret:
    raise RuntimeError("Cannot read video")

prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)

transforms = []

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Find feature points
    prev_pts = cv2.goodFeaturesToTrack(
        prev_gray,
        maxCorners=200,
        qualityLevel=0.01,
        minDistance=30
    )

    if prev_pts is None:
        transforms.append([0, 0, 0])
        prev_gray = gray
        continue

    # Track points
    curr_pts, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray,
        gray,
        prev_pts,
        None
    )

    if curr_pts is not None:

        good_old = prev_pts[status.flatten() == 1]
        good_new = curr_pts[status.flatten() == 1]

        if len(good_old) >= 8:

            M, _ = cv2.estimateAffinePartial2D(
                good_old,
                good_new
            )

            if M is not None:

                dx = M[0, 2]
                dy = M[1, 2]

                da = np.arctan2(
                    M[1, 0],
                    M[0, 0]
                )

                transforms.append([dx, dy, da])

            else:
                transforms.append([0, 0, 0])

        else:
            transforms.append([0, 0, 0])

    else:
        transforms.append([0, 0, 0])

    prev_gray = gray


cap.release()


# ---------------------------------
# SMOOTH CAMERA TRAJECTORY
# ---------------------------------

transforms = np.array(transforms)

trajectory = np.cumsum(
    transforms,
    axis=0
)

smooth_trajectory = trajectory.copy()

for i in range(3):
    smooth_trajectory[:, i] = smooth(
        trajectory[:, i],
        radius
    )

difference = smooth_trajectory - trajectory

transforms_smooth = transforms + difference


# ---------------------------------
# SECOND PASS: STABILIZE VIDEO
# ---------------------------------

cap = cv2.VideoCapture(video)

# Read first frame
ret, first = cap.read()

for i in range(len(transforms_smooth)):

    ret, frame = cap.read()

    if not ret:
        break

    dx, dy, da = transforms_smooth[i]

    # Create transformation matrix
    M = np.array([
        [np.cos(da), -np.sin(da), dx],
        [np.sin(da),  np.cos(da), dy]
    ])

    # Stabilize
    stabilized = cv2.warpAffine(
        frame,
        M,
        (w, h),
        borderMode=cv2.BORDER_REPLICATE
    )

    # ---------------------------------
    # SHOW SIDE BY SIDE
    # ---------------------------------

    cv2.putText(
        frame,
        "Original",
        (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    cv2.putText(
        stabilized,
        "Stabilized",
        (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    comparison = np.hstack((
        frame,
        stabilized
    ))

    # Resize for display
    comparison = cv2.resize(
        comparison,
        (1280, 360)
    )

    cv2.imshow(
        "Original | Stabilized",
        comparison
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()